<h4><mark>GROUP 264 - Members with Contribution Details </mark></h4>

<table border="1" style="border-collapse: collapse; text-align: center;">
  <tr>
    <th>S.NO</th>
    <th>NAME</th>
    <th>BITS ID</th>
    <th>CONTRIBUTION %</th>
  </tr>
  <tr>
    <td>1</td>
    <td>Nalla Manoj Kumar</td>
    <td>2024AC05141</td>
    <td>100 %</td>
  </tr>
  <tr>
    <td>2</td>
    <td>D Mallikarjuna Reddy</td>
    <td>2024AC05653</td>
    <td>100 %</td>
  </tr>
  <tr>
    <td>3</td>
    <td>Sajjala Ashok Reddy</td>
    <td>2024AC05829</td>
    <td>100 %</td>
  </tr>
  <tr>
    <td>4</td>
    <td>Adurti Sai Venkatesh</td>
    <td>2024AC05968</td>
    <td>100 %</td>
  </tr>
  <tr>
    <td>5</td>
    <td>M Satish Kumar</td>
    <td>2024AD05320</td>
    <td>100 %</td>
  </tr>
</table>


## PART-A: Gaming

### TicTacToe game 

##### 1) Setup: board

In [1]:
SIZE = 5          # board is 5 rows x 5 cols
WIN_LEN = 4       # need 4 in a row to win
EMPTY = '.'       # empty cell symbol

In [2]:
def make_board():

    return [[EMPTY for _ in range(SIZE)] for _ in range(SIZE)]

In [3]:
def show_board(b):
    print("\n   " + " ".join(str(c+1) for c in range(SIZE)))
    print("  +" + "--"*SIZE + "-+")
    for r in range(SIZE):
        print(f"{r+1} | " + " ".join(b[r]) + " |")
    print("  +" + "--"*SIZE + "-+\n")

In [4]:
def empty_cells(b):
    out = []
    for r in range(SIZE):
        for c in range(SIZE):
            if b[r][c] == EMPTY:
                out.append((r,c))
    return out

##### 2) All “windows of 4” and winner check


We must check every possible chunk of 4 cells in:
 - rows
 - columns
 - diagonals (down-right ↘ and up-right ↗)

There are exactly 28 such windows on a 5x5 board:
   - rows:    5 rows * 2 windows each  = 10
   - cols:    5 cols * 2 windows each  = 10
   - diag - downwards : 2 * 2 = 4
   - diag - upwards : 2 * 2 = 4
   - total = 10 + 10 + 4 + 4 = 28


In [5]:
def four_windows():
    # rows
    for r in range(SIZE):
        for c in range(SIZE - WIN_LEN + 1):
            yield [(r, c+i) for i in range(WIN_LEN)]
    # cols
    for r in range(SIZE - WIN_LEN + 1):
        for c in range(SIZE):
            yield [(r+i, c) for i in range(WIN_LEN)]
    # diag down-right ↘
    for r in range(SIZE - WIN_LEN + 1):
        for c in range(SIZE - WIN_LEN + 1):
            yield [(r+i, c+i) for i in range(WIN_LEN)]
    # diag up-right ↗
    for r in range(WIN_LEN - 1, SIZE):
        for c in range(SIZE - WIN_LEN + 1):
            yield [(r-i, c+i) for i in range(WIN_LEN)]

In [6]:
def who_won(b):
    for inds in four_windows():
        cells = [b[r][c] for (r,c) in inds]
        # if the window looks like ['X','X','X','X'] => X wins
        if cells.count('X') == WIN_LEN:
            return 'X'
        if cells.count('T') == WIN_LEN:
            return 'T'
    return None

In [7]:
def board_full(b):
    for r in range(SIZE):
        for c in range(SIZE):
            if b[r][c] == EMPTY:
                return False
    return True

##### 3) Static evaluation function

 Minimax can't always search to the very end (too many moves).
 
 So when we stop at some depth, we still need a way to decide if a position looks "good" or "bad" for the AI.

 WHAT we do:
 - Scan all 28 windows of length 4.
 - For each window (4 cells), count:
       a = how many AI symbols
       h = how many human symbols
 - If both a>0 and h>0, that window is "blocked" -> contributes 0.
 - Otherwise add a positive weight for AI-only windows, subtract for human-only.

 Weights (simple but very effective):
   - 1 in a window ->   1
   - 2 in a window ->   5
   - 3 in a window ->  50
   - 4 in a window -> 100000  (but real wins are handled separately)

 Tiny center bonus (tie-breaker): +1 for AI 'X', -1 for human 'T'
 at the five central-ish cells: (2,2), (2,1), (2,3), (1,2), (3,2)  [0-based]

In [8]:
def evaluate(b, ai, human):
    # terminal states are MUCH more important than heuristic
    w = who_won(b)
    if w == ai:
        return 10**9        # very large positive if AI already won
    if w == human:
        return -10**9       # very large negative if human already won
    if board_full(b):
        return 0            # draw-ish position

    weights = {1:1, 2:5, 3:50, 4:100000}
    score = 0

    # look at every 4-cell window and score it
    for inds in four_windows():
        vals = [b[r][c] for (r,c) in inds]
        a = vals.count(ai)
        h = vals.count(human)
        if a > 0 and h > 0:
            # if the window has both players, nobody can make 4 *in this window*
            # so it doesn't help either side directly
            continue
        if a > 0:
            score += weights[a]   # good for AI
        elif h > 0:
            score -= weights[h]   # good for human (bad for AI)

    # tiny center bias (helps break ties)
    center_cells = [(2,2), (2,1), (2,3), (1,2), (3,2)]
    for (r,c) in center_cells:
        if b[r][c] == ai:
            score += 1
        elif b[r][c] == human:
            score -= 1

    return score

##### 4) Minimax with alpha–beta

 WHY Minimax?
 - We assume the opponent tries to MINIMIZE our evaluation.
 - We try to MAXIMIZE it.

 WHY alpha-beta?
 - It cuts away branches that cannot change the final choice,
   making it faster without changing the result.

 HOW:
 - depth: how many plies (half-moves) ahead we look.
 - maximizing=True when it's AI's turn, False when it's human's turn.


In [9]:
from math import inf

def minimax(b, depth, alpha, beta, maximizing, ai, human):
    win = who_won(b)
    if depth == 0 or win is not None or board_full(b):
        # stop searching here and return the evaluation number
        return evaluate(b, ai, human), None

    best_move = None
    moves = empty_cells(b)

    # a tiny move ordering: prefer center-ish moves first
    # (This often leads to more pruning; still simple.)
    center_r, center_c = SIZE//2, SIZE//2
    moves.sort(key=lambda rc: abs(rc[0]-center_r) + abs(rc[1]-center_c))

    if maximizing:  # AI's turn
        best_score = -inf
        for (r,c) in moves:
            b[r][c] = ai              # try the move
            sc, _ = minimax(b, depth-1, alpha, beta, False, ai, human)
            b[r][c] = EMPTY           # undo the move
            if sc > best_score:
                best_score = sc
                best_move = (r,c)
            # update alpha (best score we can guarantee so far)
            if best_score > alpha:
                alpha = best_score
            # pruning: if alpha >= beta, the minimizing player won't allow this path
            if alpha >= beta:
                break
        return best_score, best_move
    else:           # Human's turn (tries to make score small)
        best_score = inf
        for (r,c) in moves:
            b[r][c] = human
            sc, _ = minimax(b, depth-1, alpha, beta, True, ai, human)
            b[r][c] = EMPTY
            if sc < best_score:
                best_score = sc
                best_move = (r,c)
            if best_score < beta:
                beta = best_score
            if alpha >= beta:
                break
        return best_score, best_move

##### 5) Input helpers

In [10]:
def ask_int(msg, lo, hi):
    while True:
        raw = input(msg).strip()
        if raw.isdigit():
            x = int(raw)
            if lo <= x <= hi:
                return x
        print(f"Please type a number between {lo} and {hi}.")

In [11]:
def ask_choice(msg, choices):
    choices = [c.upper() for c in choices]
    while True:
        x = input(f"{msg} ({'/'.join(choices)}): ").strip().upper()
        if x in choices:
            return x
        print(f"Please enter one of: {', '.join(choices)}")

##### 6) Game loop (human vs AI)

In [13]:
def play_game():
    print("\n=== 5x5 Tic-Tac-Toe (4 in a row) — Human vs AI ===")
    print("Marks are 'X' and 'T'. First to get FOUR in a straight line wins.\n")

    # depth 2..5 is reasonable; more is stronger but slower
    depth = ask_int("Pick AI depth (2-5 is fine): ", 1, 6)

    human = ask_choice("Pick your mark", ["X","T"])
    ai = "T" if human == "X" else "X"

    first = ask_choice("Who moves first? Human(H) or AI(A)", ["H","A"])

    board = make_board()
    show_board(board)

    turn = first  # 'H' or 'A'

    # Loop until win or draw
    while True:
        w = who_won(board)
        if w is not None:
            show_board(board)
            if w == human:
                print(" You WIN!")
            else:
                print(" AI WINS!")
            break

        if board_full(board):
            show_board(board)
            print("It is a DRAW. Good game!")
            break

        if turn == 'H':
            print("Your move:")
            while True:
                r = ask_int("Row (1..5): ", 1, 5) - 1
                c = ask_int("Col (1..5): ", 1, 5) - 1
                if board[r][c] == EMPTY:
                    board[r][c] = human
                    break
                else:
                    print("That cell is taken. Try a different spot.")
            show_board(board)
            turn = 'A'
        else:
            print("AI is thinking...")
            # run minimax to choose a move
            _, move = minimax(board, depth, -inf, inf, True, ai, human)
            if move is None:  # should only happen at terminal
                print("No legal moves. Stopping.")
                break
            r,c = move
            board[r][c] = ai
            print(f"AI plays at row {r+1}, col {c+1}")
            show_board(board)
            turn = 'H'

# actually run the game when file is executed
if __name__ == "__main__":
    play_game()



=== 5x5 Tic-Tac-Toe (4 in a row) — Human vs AI ===
Marks are 'X' and 'T'. First to get FOUR in a straight line wins.

Pick AI depth (2-5 is fine): 2
Pick your mark (X/T): 3
Please enter one of: X, T
Pick your mark (X/T): X
Who moves first? Human(H) or AI(A) (H/A): AI
Please enter one of: H, A
Who moves first? Human(H) or AI(A) (H/A): A

   1 2 3 4 5
  +-----------+
1 | . . . . . |
2 | . . . . . |
3 | . . . . . |
4 | . . . . . |
5 | . . . . . |
  +-----------+

AI is thinking...
AI plays at row 3, col 3

   1 2 3 4 5
  +-----------+
1 | . . . . . |
2 | . . . . . |
3 | . . T . . |
4 | . . . . . |
5 | . . . . . |
  +-----------+

Your move:
Row (1..5): 2
Col (1..5): 3

   1 2 3 4 5
  +-----------+
1 | . . . . . |
2 | . . X . . |
3 | . . T . . |
4 | . . . . . |
5 | . . . . . |
  +-----------+

AI is thinking...
AI plays at row 2, col 2

   1 2 3 4 5
  +-----------+
1 | . . . . . |
2 | . T X . . |
3 | . . T . . |
4 | . . . . . |
5 | . . . . . |
  +-----------+

Your move:
Row (1..5): 3
Col

## Tic-Tac-Toe —  Tkinter GUI 

In [13]:
import tkinter as tk
from math import inf

SIZE, WIN = 5, 4

# ---------- Core game logic (same rules, compact) ----------
def all_windows():
    # yield lists of 4 (r,c) positions for all row/col/diag windows
    for r in range(SIZE):
        for c in range(SIZE - WIN + 1):
            yield [(r, c+i) for i in range(WIN)]
    for r in range(SIZE - WIN + 1):
        for c in range(SIZE):
            yield [(r+i, c) for i in range(WIN)]
    for r in range(SIZE - WIN + 1):
        for c in range(SIZE - WIN + 1):
            yield [(r+i, c+i) for i in range(WIN)]
    for r in range(WIN-1, SIZE):
        for c in range(SIZE - WIN + 1):
            yield [(r-i, c+i) for i in range(WIN)]

FOUR_WINDOWS = tuple(all_windows())

def check_winner(board):
    # returns ('X' or 'T', winning_cells) or (None, None)
    for inds in FOUR_WINDOWS:
        vals = [board[r][c] for r,c in inds]
        if vals.count('X') == WIN: return 'X', inds
        if vals.count('T') == WIN: return 'T', inds
    return None, None

def board_full(board):
    return all(cell != '.' for row in board for cell in row)

def empty_cells(board):
    return [(r,c) for r in range(SIZE) for c in range(SIZE) if board[r][c]=='.']

def evaluate(b, ai, human):
    # terminal priority
    w,_ = check_winner(b)
    if w == ai: return 10**9
    if w == human: return -10**9
    if board_full(b): return 0
    # heuristic: sum over all 4-cell windows
    weights = {1:1, 2:5, 3:50, 4:100000}
    s = 0
    for inds in FOUR_WINDOWS:
        vals = [b[r][c] for r,c in inds]
        a = vals.count(ai); h = vals.count(human)
        if a and h: continue
        if a: s += weights[a]
        elif h: s -= weights[h]
    # tiny center bias to break ties (5 middle-ish cells)
    for (r,c) in [(2,2),(2,1),(2,3),(1,2),(3,2)]:
        if b[r][c]==ai: s+=1
        elif b[r][c]==human: s-=1
    return s

def minimax(b, depth, alpha, beta, maxing, ai, human):
    w,_ = check_winner(b)
    if depth==0 or w or board_full(b): return evaluate(b,ai,human), None
    # center-first move ordering helps pruning
    center = (SIZE//2, SIZE//2)
    moves = sorted(empty_cells(b), key=lambda rc: abs(rc[0]-center[0])+abs(rc[1]-center[1]))
    if maxing:
        best, mv = -inf, None
        for r,c in moves:
            b[r][c]=ai
            sc,_ = minimax(b, depth-1, alpha, beta, False, ai, human)
            b[r][c]='.'
            if sc>best: best, mv = sc,(r,c)
            alpha = max(alpha, best)
            if alpha>=beta: break
        return best, mv
    else:
        best, mv = inf, None
        for r,c in moves:
            b[r][c]=human
            sc,_ = minimax(b, depth-1, alpha, beta, True, ai, human)
            b[r][c]='.'
            if sc<best: best, mv = sc,(r,c)
            beta = min(beta, best)
            if alpha>=beta: break
        return best, mv

# ---------- GUI ----------
class App:
    def __init__(self, root):
        self.root = root
        root.title("5×5 Tic-Tac-Toe (4-in-a-row) — Minimax")
        root.configure(bg="#0b1220")
        # headline
        self.h1 = tk.Label(root, text="5×5 Tic-Tac-Toe (4-in-a-row)", fg="#e5e7eb", bg="#0b1220",
                           font=("Segoe UI", 18, "bold"))
        self.h1.pack(pady=(12,2))
        self.sub = tk.Label(root, text="Human vs AI • Minimax with alpha–beta", fg="#9ca3af", bg="#0b1220",
                            font=("Segoe UI", 10))
        self.sub.pack(pady=(0,10))

        # controls bar
        bar = tk.Frame(root, bg="#0b1220")
        bar.pack()
        self.depth = tk.IntVar(value=3)
        self.human_mark = tk.StringVar(value="X")
        self.first = tk.StringVar(value="H")
        self.status = tk.StringVar(value="Pick options and click New Game")

        self._mk_dropdown(bar, "Difficulty (depth):", self.depth, [2,3,4,5])
        self._mk_dropdown(bar, "Your mark:", self.human_mark, ["X","T"])
        self._mk_dropdown(bar, "First move:", self.first, ["H","A"])
        tk.Button(bar, text="New Game", command=self.new_game, bg="#1d4ed8", fg="white",
                  relief="flat", font=("Segoe UI",10,"bold"), padx=10, pady=6).pack(side="left", padx=8)

        # status line
        tk.Label(root, textvariable=self.status, fg="#93c5fd", bg="#0b1220",
                 font=("Segoe UI", 11, "bold")).pack(pady=8)

        # board frame
        self.grid = tk.Frame(root, bg="#0b1220")
        self.grid.pack(pady=6)
        self.buttons = [[None]*SIZE for _ in range(SIZE)]

        # init board
        self.board = [['.']*SIZE for _ in range(SIZE)]
        self.ai = "T"
        self.human = "X"
        self.locked = False
        self._build_board()
        self.new_game()

    def _mk_dropdown(self, parent, label, var, options):
        f = tk.Frame(parent, bg="#0b1220"); f.pack(side="left", padx=8)
        tk.Label(f, text=label, fg="#cbd5e1", bg="#0b1220", font=("Segoe UI",9)).pack(anchor="w")
        om = tk.OptionMenu(f, var, *options)
        om.config(font=("Segoe UI",10), bg="#111827", fg="#e5e7eb"), om["menu"].config(bg="white")
        om.pack()

    def _build_board(self):
        for r in range(SIZE):
            for c in range(SIZE):
                b = tk.Button(self.grid, text=" ", width=4, height=2,
                              font=("Segoe UI", 16, "bold"),
                              fg="#e5e7eb", bg="#1f2937", activebackground="#334155",
                              relief="flat",
                              command=lambda rr=r,cc=c: self.human_move(rr,cc))
                b.grid(row=r, column=c, padx=3, pady=3)
                self.buttons[r][c] = b

    def new_game(self):
        # reset board + UI
        self.board = [['.']*SIZE for _ in range(SIZE)]
        for r in range(SIZE):
            for c in range(SIZE):
                self.buttons[r][c].config(text=" ", bg="#1f2937", state="normal")
        self.human = self.human_mark.get()
        self.ai = "T" if self.human == "X" else "X"
        self.locked = False
        who = "You" if self.first.get()=="H" else "AI"
        self.status.set(f"New game started • {who} plays first (You={self.human}, AI={self.ai})")
        # AI starts?
        if self.first.get()=="A":
            self.root.after(200, self.ai_turn)

    def human_move(self, r, c):
        if self.locked or self.board[r][c] != '.': return
        self.place(r, c, self.human)
        self.after_move(next_turn="AI")

    def ai_turn(self):
        if self.locked: return
        self.status.set("AI is thinking...")
        self.root.update_idletasks()
        _, move = minimax(self.board, self.depth.get(), -inf, inf, True, self.ai, self.human)
        if move:
            r,c = move
            self.place(r, c, self.ai)
        self.after_move(next_turn="HUMAN")

    def after_move(self, next_turn):
        winner, cells = check_winner(self.board)
        if winner:
            self.highlight_win(cells)
            self.status.set("You WIN!" if winner==self.human else "AI WINS!")
            self.locked = True
            self.disable_board()
            return
        if board_full(self.board):
            self.status.set("Draw. No more moves.")
            self.disable_board()
            return
        self.status.set("Your turn." if next_turn=="HUMAN" else "AI thinking…")
        if next_turn=="AI":
            self.root.after(200, self.ai_turn)

    def place(self, r, c, sym):
        self.board[r][c] = sym
        self.buttons[r][c].config(text=sym, fg="#22c55e" if sym==self.human else "#60a5fa")

    def highlight_win(self, cells):
        for r,c in cells:
            self.buttons[r][c].config(bg="#064e3b")  # deep green

    def disable_board(self):
        for r in range(SIZE):
            for c in range(SIZE):
                self.buttons[r][c].config(state="disabled")

if __name__ == "__main__":
    tk.Tk.report_callback_exception = lambda *args: None  # keep UI calm on minor delays
    App(tk.Tk()).root.mainloop()
